In [ ]:
# -------------------------------------------------------------------
# Repo-specific imports
# -------------------------------------------------------------------
# import sys 
# print(sys.path)
# from MAST_benchmark.globals import REPO_ROOT, TOOLS_DIR

import pandas as pd

from MAST_benchmark.tools.utils import get_device
from MAST_benchmark.data_split import get_train_test_val_shots

# Set device
device = get_device()
# print(f"Using device: {device}\n")

train_shots_, test_shots_, val_shots_ = get_train_test_val_shots(
    max_index=None
)

local_flag = True

In [96]:
# Function to compute z-score and IQR outliers per variable/stat
def compute_z_and_iqr(group):
    # --- Z-score ---
    group_mean = group['value'].mean()
    group_std = group['value'].std()
    group['z_score'] = (group['value'] - group_mean) / group_std
    group['outlier_z_6'] = group['z_score'].abs() > 6
    group['outlier_z_12'] = group['z_score'].abs() > 12
    return group

In [104]:
# df = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_1.csv")
# print(df.head())

files = [
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_1.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_2.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_3.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_4.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_5.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_6.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_7.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_8.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_9.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_10.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_11.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_12.csv",
    "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_13_other_way.csv",
]

# files = [
#     "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_val.csv"
# ]

# files = [
#     "/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_test.csv"
# ]

dfs = [pd.read_csv(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True)
print(df_all.shape)
# print(df_all.head())

dup_count = df_all.duplicated().sum()
print("Duplicate rows:", dup_count)
df = df_all.sort_values(by="shot_idx")

(9259, 80)
Duplicate rows: 0


In [105]:
# Identify measurement columns
measure_cols = [c for c in df_all.columns if c not in ['shot_idx','shot_id']]

# Melt into long format
df_long = df_all.melt(
    id_vars=['shot_idx','shot_id'],
    value_vars=measure_cols,
    var_name='variable_stat',
    value_name='value'
)

# Split variable_stat into variable and stat
df_long[['variable','stat']] = df_long['variable_stat'].str.rsplit('_', n=1, expand=True)
df_long = df_long.drop(columns='variable_stat')

# Optional: reorder columns
df_long = df_long[['shot_idx','shot_id','variable','stat','value']]

print(df_long.head())


   shot_idx  shot_id                  variable  stat     value
0         0    21719  magnetics-flux_loop_flux  mean -0.337641
1         1    29562  magnetics-flux_loop_flux  mean -0.333371
2         2    27570  magnetics-flux_loop_flux  mean -0.302958
3         3    25984  magnetics-flux_loop_flux  mean -0.158291
4         4    26995  magnetics-flux_loop_flux  mean -0.240361


In [106]:
values_to_remove = ["soft_x_rays-horizontal_cam_lower", "soft_x_rays-horizontal_cam_upper"]
df_long = df_long[~df_long["variable"].isin(values_to_remove)]
len(df_long)

685166

In [107]:
len( df_long['variable'].unique() )

37

In [108]:
# add soft_xrays

df_xr = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_soft_x_rays.csv")
# df_xr = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_soft_x_rays_val.csv")
# df_xr = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/shot_statistics_soft_x_rays_test.csv")

dup_count = df_xr.duplicated().sum()
print("Duplicate rows:", dup_count)
df_xr = df_xr.sort_values(by="shot_idx")

# Identify measurement columns
measure_cols = [c for c in df_xr.columns if c not in ['shot_idx','shot_id']]

# Melt into long format
df_xr = df_xr.melt(
    id_vars=['shot_idx','shot_id'],
    value_vars=measure_cols,
    var_name='variable_stat',
    value_name='value'
)

# Split variable_stat into variable and stat
df_xr[['variable','stat']] = df_xr['variable_stat'].str.rsplit('_', n=1, expand=True)
df_xr = df_xr.drop(columns='variable_stat')

# Optional: reorder columns
df_xr = df_xr[['shot_idx','shot_id','variable','stat','value']]

print(df_xr.head())
print(len(df_xr))

Duplicate rows: 0
   shot_idx  shot_id                          variable  stat     value
0         0    21719  soft_x_rays-horizontal_cam_lower  mean  0.000700
1         1    29562  soft_x_rays-horizontal_cam_lower  mean       NaN
2         2    27570  soft_x_rays-horizontal_cam_lower  mean  0.005428
3         3    25984  soft_x_rays-horizontal_cam_lower  mean  0.003782
4         4    26995  soft_x_rays-horizontal_cam_lower  mean  0.008759
37036


In [109]:
df_train = pd.concat([df_long, df_xr]).sort_values(by=["shot_idx", "variable", "stat"])
df_train.to_csv("shot_statistics_train.csv", index=False)
# Apply per variable/stat
df_stat_out = df_train.groupby(['variable','stat'], group_keys=False).apply(compute_z_and_iqr)
df_stat_out.to_csv("shot_statistics_and_outlier_train.csv", index=False)

# df_val = pd.concat([df_long, df_xr]).sort_values(by=["shot_idx", "variable", "stat"])
# df_val.to_csv("shot_statistics_val.csv", index=False)
# # Apply per variable/stat
# df_stat_out = df_val.groupby(['variable','stat'], group_keys=False).apply(compute_z_and_iqr)
# df_stat_out.to_csv("shot_statistics_and_outlier_val.csv", index=False)

# df_test = pd.concat([df_long, df_xr]).sort_values(by=["shot_idx", "variable", "stat"])
# df_test.to_csv("shot_statistics_test.csv", index=False)
# # Apply per variable/stat
# df_stat_out = df_test.groupby(['variable','stat'], group_keys=False).apply(compute_z_and_iqr)
# df_stat_out.to_csv("shot_statistics_and_outlier_test.csv", index=False)




/tmp/ipykernel_2483959/917006481.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_stat_out = df_train.groupby(['variable','stat'], group_keys=False).apply(compute_z_and_iqr)


In [112]:
import pandas as pd

# Compute means
df_mean = df_stat_out[df_stat_out['stat']=='mean']
mean_all = df_mean.groupby('variable')['value'].mean().rename('mean_all')
mean_no_outliers_z6 = df_mean[~df_mean['outlier_z_6']].groupby('variable')['value'].mean().rename('mean_no_outliers_z6')
mean_no_outliers_z12 = df_mean[~df_mean['outlier_z_12']].groupby('variable')['value'].mean().rename('mean_no_outliers_z12')

# Compute std
df_std = df_stat_out[df_stat_out['stat']=='std']
std_all = df_std.groupby('variable')['value'].mean().rename('std_all')
std_no_outliers_z6 = df_std[~df_std['outlier_z_6']].groupby('variable')['value'].mean().rename('std_no_outliers_z6')
std_no_outliers_z12 = df_std[~df_std['outlier_z_12']].groupby('variable')['value'].mean().rename('std_no_outliers_z12')

# Combine into one table
stats_combined = pd.concat([mean_all, mean_no_outliers_z6, mean_no_outliers_z12,
                           std_all, std_no_outliers_z6, std_no_outliers_z12], axis=1).reset_index()


In [113]:

import yaml

# Load CSV
df = stats_combined

final_dict = {}

for _, row in df.iterrows():
    var = row["variable"]

    final_dict[var] = {
        "mean": {
            "all": row["mean_all"],
            "no_outliers_z6": row["mean_no_outliers_z6"],
            "no_outliers_z12": row["mean_no_outliers_z12"]
        },
        "std": {
            "all": row["std_all"],
            "no_outliers_z6": row["std_no_outliers_z6"],
            "no_outliers_z12": row["std_no_outliers_z12"]
        }
    }

with open("mean_std_train.yaml", "w") as f:
    yaml.dump(final_dict, f, sort_keys=False)



In [114]:
final_dict

{'equilibrium-beta_normal': {'mean': {'all': 0.9444201328197982,
   'no_outliers_z6': 1.0070692705319584,
   'no_outliers_z12': 0.9807728804388701},
  'std': {'all': 0.8485818423539658,
   'no_outliers_z6': 0.8104075309014818,
   'no_outliers_z12': 0.8279526568977954}},
 'equilibrium-beta_pol': {'mean': {'all': 0.1980151193255034,
   'no_outliers_z6': 0.21066049192496245,
   'no_outliers_z12': 0.20540090847261136},
  'std': {'all': 0.18494559238894906,
   'no_outliers_z6': 0.17629562991584707,
   'no_outliers_z12': 0.18152670926513984}},
 'equilibrium-beta_tor': {'mean': {'all': 2.8994700843181214,
   'no_outliers_z6': 2.386993916157026,
   'no_outliers_z12': 2.3626524301283247},
  'std': {'all': 2.42070816707415,
   'no_outliers_z6': 1.991196415832752,
   'no_outliers_z12': 1.9973836613014615}},
 'equilibrium-bphi_rmag': {'mean': {'all': -0.5099775902438537,
   'no_outliers_z6': -0.5086097377287538,
   'no_outliers_z12': -0.509652766178864},
  'std': {'all': 0.05175379778605441,
   'n

In [115]:
train_csv = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_train.csv")
val_csv = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_val.csv")
test_csv = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_test.csv")



In [143]:
# train_csv [ (train_csv['stat']== 'mean') & train_csv['outlier_z_12'] ]


import pandas as pd

# Example: load your CSV
df_train = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_train.csv")
df_val = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_val.csv")
df_test = pd.read_csv("/home/ir-rous1/rds/rds-ukaea-ap002-mOlK9qn0PlQ/ir-rous1/output/fairmast-data-preprocessing/artifacts/stats_mean_std/shot_statistics_and_outlier_test.csv")


# df = pd.concat([df_train, df_val, df_test])?
# Filter for rows where stat is 'mean' and at least one outlier flag is True
outlier_rows = df[(df["stat"] == "mean") & (df["outlier_z_12"])]

# Build dictionary: shot_id -> list of variables
outlier_dict = (
    outlier_rows.groupby("shot_id")["variable"]
    .apply(list)
    .to_dict()
)

print(outlier_dict)


{12004: ['interferometer-n_e_line'], 12139: ['equilibrium-x_point_r'], 12140: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12142: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12143: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12144: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12145: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12146: ['equilibrium-x_point_r'], 12147: ['equilibrium-x_point_r'], 12148: ['equilibrium-x_point_r'], 12150: ['equilibrium-x_point_r'], 12151: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12158: ['equilibrium-x_point_r'], 12161: ['equilibrium-x_point_r'], 12162: ['equilibrium-x_point_r'], 12163: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12164: ['equilibrium-x_point_r'], 12165: ['equilibrium-x_point_r'], 12166: ['equilibrium-x_point_r'], 12168: ['equilibrium-x_point_z'], 12169: ['equilibrium-x_point_r', 'equilibrium-x_point_z'], 12170: ['equilibrium-x_point_r'], 12171: ['equilibrium-x_point_r'], 12172: ['equili

In [144]:
len(outlier_dict)

120

In [145]:
with open("dict_outlier_metadata.yaml", "w") as f_:
    yaml.dump(outlier_dict, f_, sort_keys=False)